Question: Which taxi type was the most used in the first half of 2024?

# Obtain and preview data in general

In [1]:
## Import data
# Create a data folder.
import os
import urllib
os.makedirs('data_folder', exist_ok=True)
# Define the URL and file path.
yellow = ["https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
          "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet",
          "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet",
          "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-04.parquet",
          "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-05.parquet",
          "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-06.parquet",
          ]

green = ["https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet",
         "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-02.parquet",
         "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-03.parquet",
         "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-04.parquet",
         "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-05.parquet",
         "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-06.parquet",
        ]

#creates function to downloads the parquet
def download_all(urls):
    for url in urls:
        filename = url.split("/")[-1]  #using the last string (e.g. green_tripdata_2024-03.parquet)
        filepath = os.path.join("data_folder", filename)

        if os.path.exists(filepath):
            print(f"Skipping download. {filename} already exists.")
            continue

        print(f"Downloading {filename}...")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Saved to: {filepath}")
        except Exception as e:
            print(f"Download failed for {url}. Error: {e}")

#download the parquet
download_all(yellow)
download_all(green)


Saved to: data_folder/yellow_tripdata_2024-01.parquet
Saved to: data_folder/yellow_tripdata_2024-02.parquet
Saved to: data_folder/yellow_tripdata_2024-03.parquet
Saved to: data_folder/yellow_tripdata_2024-04.parquet
Saved to: data_folder/yellow_tripdata_2024-05.parquet
Saved to: data_folder/yellow_tripdata_2024-06.parquet
Saved to: data_folder/green_tripdata_2024-01.parquet
Saved to: data_folder/green_tripdata_2024-02.parquet
Saved to: data_folder/green_tripdata_2024-03.parquet
Saved to: data_folder/green_tripdata_2024-04.parquet
Saved to: data_folder/green_tripdata_2024-05.parquet
Saved to: data_folder/green_tripdata_2024-06.parquet


In [2]:
## Inspect data's columns
import pyarrow.parquet as pq
import os

for f in sorted(os.listdir("data_folder")):
    path = os.path.join("data_folder", f)
    schema = pq.read_schema(path)
    print(f"{f}: {len(schema)} columns")
    print(schema.names)
    print("---")

green_tripdata_2024-01.parquet: 20 columns
['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge']
---
green_tripdata_2024-02.parquet: 20 columns
['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge']
---
green_tripdata_2024-03.parquet: 20 columns
['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra

In [3]:
##Compare both columns
schemas = {}
for f in sorted(os.listdir("data_folder")):
    path = os.path.join("data_folder", f)
    schemas[f] = set(pq.read_schema(path).names)

# check if all yellow files have the same columns
yellow_files = [f for f in schemas if f.startswith("yellow")]
all_yellow = set.union(*(schemas[f] for f in yellow_files))

# do the same with green
green_files = [f for f in schemas if f.startswith("green")]
all_green = set.union(*(schemas[f] for f in green_files))

print("Yellow-only columns:", all_yellow - all_green)
print("Green-only columns:", all_green - all_yellow)

Yellow-only columns: {'tpep_dropoff_datetime', 'tpep_pickup_datetime', 'Airport_fee'}
Green-only columns: {'ehail_fee', 'lpep_dropoff_datetime', 'lpep_pickup_datetime', 'trip_type'}


In [4]:
## Convert paquet to df
import pandas as pd

# Load all yellow parquet files, sort them, and read into separate DataFrames
yellow_dfs = [pd.read_parquet(f"data_folder/{f}") for f in sorted(yellow_files)]
#combine all yellow dataFrame into a single DataFrame
yellow_df = pd.concat(yellow_dfs, ignore_index=True)

#do the same for green
green_dfs = [pd.read_parquet(f"data_folder/{f}") for f in sorted(green_files)]
green_df = pd.concat(green_dfs, ignore_index=True)

# Clean + Modifiy the data

##Helper function

In [5]:
##function to check dataframe structure
def summarize(df, name):
    print(f"=== {name} ===")
    print("Shape:", df.shape) #Shape: (row, column)
    print("\nDtypes:")
    print(df.dtypes) #print list of datatype and variablee=
    print("\nNull counts:")
    null_counts = df.isnull().sum() #counting null
    print(null_counts[null_counts > 0]), #ignore column with zero null
    print("=================================\n")

## Inspect yellow data

In [6]:
summarize(yellow_df, "Yellow")
## running 'yellow_df.describe(include='number').T' will crash due to the size of the file, we will switch to duckDB
import duckdb

con = duckdb.connect()

# Query directly across all yellow files
yellow_summary = con.execute("""
    SELECT
        COUNT(*) as n_rows,
        AVG(trip_distance) as avg_trip_distance,
        MIN(trip_distance) as min_trip_distance,
        MAX(trip_distance) as max_trip_distance,
        AVG(fare_amount) as avg_fare,
        MIN(fare_amount) as min_fare,
        MAX(fare_amount) as max_fare
    FROM 'data_folder/yellow_tripdata_2024-*.parquet'
""").df()

print(yellow_summary)

=== Yellow ===
Shape: (20332093, 19)

Dtypes:
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
dtype: object

Null counts:
passenger_count         1975985
RatecodeID              1975985
store_and_fwd_flag      1975985
congestion_surcharge    1975985
Airport_fee             1975985
dtype:

In [7]:
##Next we have to check the row that breaks these condition
#-passenger_count must exist or more than zero
#-pickup time must happen before dropoff
#-fare amount can not be negative or zero
yellow_invalid_rows = con.execute("""
    SELECT
        *
    FROM 'data_folder/yellow_tripdata_2024-*.parquet'
    WHERE
        passenger_count IS NULL OR
        passenger_count <= 0 OR
        tpep_pickup_datetime >= tpep_dropoff_datetime OR
        fare_amount <= 0
""").df()

print("Rows which break the condition:")
display(yellow_invalid_rows)
#we will remove time as the data may be invalid

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows which break the condition:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.00,1,N,246,231,2,25.40,3.50,0.5,0.00,0.00,1.0,30.40,2.5,0.00
1,1,2024-01-01 00:14:29,2024-01-01 00:14:29,1,0.00,1,N,236,264,2,3.00,3.50,0.5,0.00,0.00,1.0,8.00,2.5,0.00
2,2,2024-01-01 00:18:24,2024-01-01 00:30:39,1,2.16,1,N,249,232,4,-13.50,-1.00,-0.5,0.00,0.00,-1.0,-18.50,-2.5,0.00
3,1,2024-01-01 00:49:42,2024-01-01 00:58:21,0,2.90,1,N,87,79,1,14.90,3.50,0.5,5.00,0.00,1.0,24.90,2.5,0.00
4,1,2024-01-01 00:42:26,2024-01-01 01:12:01,0,16.60,2,N,132,233,1,70.00,4.25,0.5,16.53,6.94,1.0,99.22,2.5,1.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2462480,2,2024-06-30 23:07:36,2024-06-30 23:18:35,<NA>,2.51,<NA>,None,255,148,0,21.09,0.00,0.5,0.00,0.00,1.0,25.09,NaN,NaN
2462481,2,2024-06-30 23:46:07,2024-07-01 00:11:53,<NA>,7.55,<NA>,None,68,74,0,33.96,0.00,0.5,0.00,0.00,1.0,37.96,NaN,NaN
2462482,2,2024-06-30 23:18:50,2024-06-30 23:35:09,<NA>,4.25,<NA>,None,41,100,0,21.01,0.00,0.5,0.00,0.00,1.0,25.01,NaN,NaN
2462483,1,2024-06-30 23:33:36,2024-06-30 23:42:37,<NA>,1.60,<NA>,None,158,231,0,14.41,0.00,0.5,0.00,0.00,1.0,18.41,NaN,NaN


## Inspect green data



In [8]:
summarize(green_df, "Green")
# Query directly across all green files
green_summary = con.execute("""
    SELECT
        COUNT(*) as n_rows,
        AVG(trip_distance) as avg_trip_distance,
        MIN(trip_distance) as min_trip_distance,
        MAX(trip_distance) as max_trip_distance,
        AVG(fare_amount) as avg_fare,
        MIN(fare_amount) as min_fare,
        MAX(fare_amount) as max_fare
    FROM 'data_folder/green_tripdata_2024-*.parquet'
""").df()

print(green_summary)

=== Green ===
Shape: (339807, 20)

Dtypes:
VendorID                          int32
lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
store_and_fwd_flag               object
RatecodeID                      float64
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
ehail_fee                       float64
improvement_surcharge           float64
total_amount                    float64
payment_type                    float64
trip_type                       float64
congestion_surcharge            float64
dtype: object

Null counts:
store_and_fwd_flag       14219
RatecodeID               14219
passenger_count          14219
ehail_fee               339807
payme

In [9]:
##check the row that breaks these condition
#-passenger_count must exist or more than zero
#-pickup time must happen before dropoff
#-fare amount can not be negative or zero
green_invalid_df = con.execute("""
    SELECT
        *
    FROM 'data_folder/green_tripdata_2024-*.parquet'
    WHERE
        passenger_count IS NULL OR
        passenger_count <= 0 OR
        lpep_pickup_datetime >= lpep_dropoff_datetime OR
        fare_amount <= 0
""").df()

print("DataFrame of green taxi rows where passenger_count is null:")
display(green_invalid_df)
#We planned to remove them as the data may be invalid similar to yellow data

DataFrame of green taxi rows where passenger_count is null:


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,1,2024-01-01 01:28:43,2024-01-01 01:29:12,N,5,7,7,0,0.00,15.00,0.0,0.0,0.00,0.0,NaN,0.0,15.00,1,2,0.0
1,1,2024-01-01 01:52:21,2024-01-01 01:59:00,N,1,179,223,0,0.70,7.90,1.0,1.5,2.60,0.0,NaN,1.0,13.00,1,1,0.0
2,2,2024-01-01 07:54:11,2024-01-01 08:00:18,N,1,129,82,6,0.57,-7.20,0.0,-0.5,0.00,0.0,NaN,-1.0,-8.70,4,1,0.0
3,2,2024-01-01 08:56:36,2024-01-01 08:57:27,N,1,75,74,6,0.35,-3.70,0.0,-0.5,0.00,0.0,NaN,-1.0,-5.20,3,1,0.0
4,1,2024-01-01 08:24:22,2024-01-01 08:26:11,N,1,74,74,0,1.40,3.70,0.0,1.5,0.00,0.0,NaN,1.0,5.20,3,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18949,2,2024-06-30 17:07:00,2024-06-30 17:20:00,None,<NA>,225,177,<NA>,1.42,12.48,0.0,0.0,5.00,0.0,NaN,1.0,18.48,<NA>,<NA>,NaN
18950,2,2024-06-30 18:16:00,2024-06-30 18:53:00,None,<NA>,62,157,<NA>,5.63,34.56,0.0,0.0,7.01,0.0,NaN,1.0,42.57,<NA>,<NA>,NaN
18951,2,2024-06-30 19:39:00,2024-06-30 20:00:00,None,<NA>,247,151,<NA>,3.34,24.81,0.0,0.0,5.06,0.0,NaN,1.0,30.87,<NA>,<NA>,NaN
18952,2,2024-06-30 21:28:13,2024-06-30 21:28:58,None,<NA>,7,7,<NA>,0.00,12.48,0.0,0.0,0.52,0.0,NaN,1.0,14.00,<NA>,<NA>,NaN


# Implement and Compare Cleaned Data

In [10]:
import textwrap

# Custom function to get stats and counts without loading full DF into memory (or it will crash)
def get_taxi_stats(con, taxi_type='yellow'):
    """
    Gets validation statistics and cleaned row count for a given taxi type
    without loading the entire cleaned DataFrame into memory.

    Parameters:
    -----------
    con : DuckDB connection object
        Database connection for executing queries
    taxi_type : str, default='yellow'
        Specifies which taxi data to analyze ('yellow' or 'green')

    Returns:
    --------
    tuple : (validation_stats_df, cleaned_rows_count)
        - validation_stats_df: DataFrame with validation statistics
        - cleaned_rows_count: Integer count of rows passing all validation rules
    """

    # --- Set up taxi-specific column and file names ---
    if taxi_type == 'yellow':
        table_pattern = 'data_folder/yellow_tripdata_2024-*.parquet'
        pickup_col = 'tpep_pickup_datetime'
        dropoff_col = 'tpep_dropoff_datetime'

        # Define SQL WHERE clause conditions for cleaning yellow taxi data
        cleaning_conditions_str = textwrap.dedent("""
            passenger_count > 0 AND
            {pickup_col} < {dropoff_col} AND
            fare_amount > 0
        """).format(pickup_col=pickup_col, dropoff_col=dropoff_col)

        # Define SQL validation checks as SELECT expressions
        # Each check returns TRUE if the condition is valid, FALSE otherwise
        validation_checks_str = textwrap.dedent("""
            (passenger_count > 0) as valid_passenger,
            ({pickup_col} < {dropoff_col}) as valid_time_order,
            (fare_amount > 0) as valid_fare
        """).format(pickup_col=pickup_col, dropoff_col=dropoff_col)

        # Count rows that pass each individual validation rule
        validation_summary_sums = textwrap.dedent(
            """
            SUM(CASE WHEN valid_passenger THEN 1 ELSE 0 END) as passenger_count_valid,
            SUM(CASE WHEN valid_time_order THEN 1 ELSE 0 END) as time_order_valid,
            SUM(CASE WHEN valid_fare THEN 1 ELSE 0 END) as fare_valid
            """
        )
    else:  # green
        table_pattern = 'data_folder/green_tripdata_2024-*.parquet'
        pickup_col = 'lpep_pickup_datetime'
        dropoff_col = 'lpep_dropoff_datetime'

        # Define SQL WHERE clause conditions for cleaning green taxi data
        cleaning_conditions_str = textwrap.dedent("""
            passenger_count > 0 AND
            {pickup_col} < {dropoff_col} AND
            fare_amount > 0
        """).format(pickup_col=pickup_col, dropoff_col=dropoff_col)

        # Define SQL validation checks as SELECT expressions
        validation_checks_str = textwrap.dedent("""
            (passenger_count > 0) as valid_passenger,
            ({pickup_col} < {dropoff_col}) as valid_time_order,
            (fare_amount > 0) as valid_fare
        """).format(pickup_col=pickup_col, dropoff_col=dropoff_col)

        # Count rows that pass each individual validation rule
        validation_summary_sums = textwrap.dedent(
            """
            SUM(CASE WHEN valid_passenger THEN 1 ELSE 0 END) as passenger_count_valid,
            SUM(CASE WHEN valid_time_order THEN 1 ELSE 0 END) as time_order_valid,
            SUM(CASE WHEN valid_fare THEN 1 ELSE 0 END) as fare_valid
            """
        )

    # --- Execute validation query ---
    # This query evaluates each validation check across all rows without loading the entire dataset into memory
    validation_stats_df = con.execute(f"""
        WITH raw_data AS (
            SELECT
                *,
                {validation_checks_str}
            FROM '{table_pattern}'
        ),
        validation_summary AS (
            SELECT
                COUNT(*) as total_rows,
                {validation_summary_sums}
            FROM raw_data
        )
        SELECT * FROM validation_summary
    """).df()

    # --- Execute cleaned row count query ---
    # Count only rows that pass ALL three validation conditions
    # This avoids loading the full cleaned dataset into memory
    cleaned_rows_count_df = con.execute(f"""
        SELECT
            COUNT(*) as cleaned_rows_count
        FROM '{table_pattern}'
        WHERE
            {cleaning_conditions_str}
    """).df()
    # Return both validation statistics and cleaned row count
    return validation_stats_df, cleaned_rows_count_df['cleaned_rows_count'].values[0]

# ============================================================================
# MAIN SCRIPT - Analyze taxi usage for first half of 2024
# ============================================================================

# Clean yellow taxi data
print("\n--- Cleaning Yellow Taxi Data ---")
# Call function to get validation stats and cleaned row count for yellow taxis
yellow_validation_stats, yellow_cleaned_rows_count = get_taxi_stats(con, taxi_type='yellow')

# Display validation report for yellow taxi data
print(f"\n{'='*60}")
print(f"Validation Report for YELLOW Taxi Data")
print(f"{'='*60}")
print(yellow_validation_stats.to_string(index=False))
# Calculate and display retention rate (percentage of rows kept after cleaning)
print(f"\nCleaned rows: {yellow_cleaned_rows_count} / {yellow_validation_stats['total_rows'].values[0]}")
print(f"Retention rate: {100*yellow_cleaned_rows_count/yellow_validation_stats['total_rows'].values[0]:.2f}%")


# --- Clean green taxi data ---
print("\n--- Cleaning Green Taxi Data ---")
green_validation_stats, green_cleaned_rows_count = get_taxi_stats(con, taxi_type='green')

# Display validation report for green taxi data
print(f"\n{'='*60}")
print(f"Validation Report for GREEN Taxi Data")
print(f"{'='*60}")
print(green_validation_stats.to_string(index=False))
# Calculate and display retention rate (percentage of rows kept after cleaning)
print(f"\nCleaned rows: {green_cleaned_rows_count} / {green_validation_stats['total_rows'].values[0]}")
print(f"Retention rate: {100*green_cleaned_rows_count/green_validation_stats['total_rows'].values[0]:.2f}%")


# Display side-by-side comparison of raw vs cleaned rows and retention rates
print("\n--- Comparison of Cleaned Data ---")
print(f"Yellow Taxi - Total Raw Rows: {yellow_validation_stats['total_rows'].values[0]}, Cleaned Rows: {yellow_cleaned_rows_count}, Retention Rate: {100*yellow_cleaned_rows_count/yellow_validation_stats['total_rows'].values[0]:.2f}%")
print(f"Green Taxi  - Total Raw Rows: {green_validation_stats['total_rows'].values[0]}, Cleaned Rows: {green_cleaned_rows_count}, Retention Rate: {100*green_cleaned_rows_count/green_validation_stats['total_rows'].values[0]:.2f}%")

# --- Display sample of cleaned data ---
print("\n--- Preview of cleaned yellow taxi data (sample) ---")
# Fetch and display first 5 rows of cleaned yellow and green taxi data
yellow_sample_df = con.execute(f"""
    SELECT
        *
    FROM 'data_folder/yellow_tripdata_2024-*.parquet'
    WHERE
        passenger_count > 0 AND
        tpep_pickup_datetime < tpep_dropoff_datetime AND
        fare_amount > 0
    LIMIT 5
""").df()
display(yellow_sample_df)

print("\n--- Preview of cleaned green taxi data (sample) ---")
green_sample_df = con.execute(f"""
    SELECT
        *
    FROM 'data_folder/green_tripdata_2024-*.parquet'
    WHERE
        passenger_count > 0 AND
        lpep_pickup_datetime < lpep_dropoff_datetime AND
        fare_amount > 0
    LIMIT 5
""").df()
display(green_sample_df)

#Answer the question: Which taxi type was the most used?
print("\n--- Most Used Taxi Type ---")
if yellow_cleaned_rows_count > green_cleaned_rows_count:
    print(f"Yellow taxi was used more with {yellow_cleaned_rows_count} trips compared to Green taxi with {green_cleaned_rows_count} trips.")
elif green_cleaned_rows_count > yellow_cleaned_rows_count:
    print(f"Green taxi was used more with {green_cleaned_rows_count} trips compared to Yellow taxi with {yellow_cleaned_rows_count} trips.")
else:
    print("Both Yellow and Green taxis had the same number of trips.")



--- Cleaning Yellow Taxi Data ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Validation Report for YELLOW Taxi Data
 total_rows  passenger_count_valid  time_order_valid  fare_valid
   20332093             18134999.0        20325916.0  20008013.0

Cleaned rows: 17869608 / 20332093
Retention rate: 87.89%

--- Cleaning Green Taxi Data ---

Validation Report for GREEN Taxi Data
 total_rows  passenger_count_valid  time_order_valid  fare_valid
     339807               322471.0          339472.0    338405.0

Cleaned rows: 320853 / 339807
Retention rate: 94.42%

--- Comparison of Cleaned Data ---
Yellow Taxi - Total Raw Rows: 20332093, Cleaned Rows: 17869608, Retention Rate: 87.89%
Green Taxi  - Total Raw Rows: 339807, Cleaned Rows: 320853, Retention Rate: 94.42%

--- Preview of cleaned yellow taxi data (sample) ---


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0



--- Preview of cleaned green taxi data (sample) ---


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2024-01-01 00:46:55,2024-01-01 00:58:25,N,1,236,239,1,1.98,12.8,1.0,0.5,3.61,0.0,NaN,1.0,21.66,1,1,2.75
1,2,2024-01-01 00:31:42,2024-01-01 00:52:34,N,1,65,170,5,6.54,30.3,1.0,0.5,7.11,0.0,NaN,1.0,42.66,1,1,2.75
2,2,2024-01-01 00:30:21,2024-01-01 00:49:23,N,1,74,262,1,3.08,19.8,1.0,0.5,3.00,0.0,NaN,1.0,28.05,1,1,2.75
3,1,2024-01-01 00:30:20,2024-01-01 00:42:12,N,1,74,116,1,2.40,14.2,1.0,1.5,0.00,0.0,NaN,1.0,16.70,2,1,0.00
4,2,2024-01-01 00:32:38,2024-01-01 00:43:37,N,1,74,243,1,5.14,22.6,1.0,0.5,6.28,0.0,NaN,1.0,31.38,1,1,0.00



--- Most Used Taxi Type ---
Yellow taxi was used more with 17869608 trips compared to Green taxi with 320853 trips.


As we can see here, Yellow taxi was used more than Green taxi, While Yellow taxi have less retention rate than Green taxi, there is much larger amount of data for yellow taxi than Green Taxi by 2 millions to 300 thousand.